#### Iteration 7 - LLM-Based Classification Pipeline (RQ 2)

This notebook implements an LLM-based **few-shot classification pipeline** to classify power plant incidents as **Process Safety** or **Non-Process Safety**.

##### Key Improvements:
1. **Few-shot prompting** with real labeled examples
2. **Multilingual support** (English, German, Swedish, Dutch)
3. **Optimized classification criteria**
4. **Language-specific processing**

#### Process Safety Definition:
A Process Safety Incident typically involves:
- Unexpected mechanical integrity failure in a system, processing facility, or industrial plant
- Fire, explosion, rupture, or hazardous chemical leak
- Release/loss of containment of hazardous materials (oil, gas, water, chemicals)
- Equipment trips causing unplanned shutdowns (GT trips, pump trips, boiler trips)
- Events potentially catastrophic with large-scale health and environmental consequences
- Any unplanned event causing release of hazardous material outside intended operation boundaries
- Emergency shutdowns (ESD, Not-Aus, noodstop)
- Contamination events, pressure deviations, equipment damage

In [1]:
# =============================================================================
# IMPORTS AND CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
import time
import os
import gc

warnings.filterwarnings('ignore')

# Configuration
BASE_PATH = '/home/azureuser/cloudfiles/code/Users/M02555/'
DATA_PATH_BY_COUNTRY = os.path.join(BASE_PATH, 'Datasets/By_Country/')
RESULTS_PATH = os.path.join(BASE_PATH, 'Results/_iteration_7/')

# Language-specific files
LANGUAGE_FILES = {
    'English': ['RW_ACTUALS_English.csv', 'RW_ACTUALS_UK.csv'],
    'German': ['RW_ACTUALS_Germany.csv'],
    'Dutch': ['RW_ACTUALS_Netherlands.csv'],
    'Swedish': ['RW_ACTUALS_Sweden.csv']
}

# Create results directory if not exists
os.makedirs(RESULTS_PATH, exist_ok=True)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Using device: {device}")
print(f"[INFO] Data path: {DATA_PATH_BY_COUNTRY}")
print(f"[INFO] Results path: {RESULTS_PATH}")

C:\Users\shari\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Using device: cpu
[INFO] Data path: /home/azureuser/cloudfiles/code/Users/M02555/Datasets/By_Country/
[INFO] Results path: /home/azureuser/cloudfiles/code/Users/M02555/Results/_iteration_7/


In [2]:
# =============================================================================
# LOAD AND COMBINE DATA BY LANGUAGE
# =============================================================================
print("[INFO] Loading data files by language...")

all_dfs = []
for language, files in LANGUAGE_FILES.items():
    for file in files:
        file_path = os.path.join(DATA_PATH_BY_COUNTRY, file)
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, usecols=["TITLE", "CASE_DESCRIPTION", "CASE_TYPE"])
            df['LANGUAGE'] = language
            df['SOURCE_FILE'] = file
            all_dfs.append(df)
            print(f"  [OK] Loaded {file}: {len(df)} records")
        else:
            print(f"  [WARN] File not found: {file}")

# Combine all dataframes
master_df_7 = pd.concat(all_dfs, ignore_index=True)

print(f"\n[INFO] Total records: {len(master_df_7)}")
print(f"[INFO] Columns: {list(master_df_7.columns)}")
print(f"\n[INFO] Records by language:")
print(master_df_7['LANGUAGE'].value_counts())
print(f"\n[INFO] Class distribution in CASE_TYPE:")
print(master_df_7['CASE_TYPE'].value_counts())
print(f"\n[INFO] Process Safety count: {len(master_df_7[master_df_7['CASE_TYPE']=='Process Safety'])}")
master_df_7.head()

[INFO] Loading data files by language...
  [OK] Loaded RW_ACTUALS_English.csv: 7094 records
  [OK] Loaded RW_ACTUALS_UK.csv: 7019 records
  [OK] Loaded RW_ACTUALS_Germany.csv: 22836 records
  [OK] Loaded RW_ACTUALS_Netherlands.csv: 3168 records
  [OK] Loaded RW_ACTUALS_Sweden.csv: 20851 records

[INFO] Total records: 60968
[INFO] Columns: ['CASE_DESCRIPTION', 'CASE_TYPE', 'TITLE', 'LANGUAGE', 'SOURCE_FILE']

[INFO] Records by language:
LANGUAGE
German     22836
Swedish    20851
English    14113
Dutch       3168
Name: count, dtype: int64

[INFO] Class distribution in CASE_TYPE:
CASE_TYPE
Safety                              41572
Process Safety                       9650
Environment                          5134
Asset and Reputation damage/loss     2559
Operational loss                     1046
Information Security                  341
Physical Security                     105
Not classified                          8
Name: count, dtype: int64

[INFO] Process Safety count: 9650


,CASE_DESCRIPTION,CASE_TYPE,TITLE,LANGUAGE,SOURCE_FILE
0,01.07 1400hrs UFA fault hold – alarm observed...,Process Safety,WTP UFA overpressurisation,English,RW_ACTUALS_English.csv
1,"aerosol paint can failed mechanically, small 4...",Process Safety,Process Safety,English,RW_ACTUALS_English.csv
2,No.1 Bulk Sodium Hypochlorite Tank lost approx...,Process Safety,PSNH - Loss of Hypo from No.1 Bulk Sodium Hypo...,English,RW_ACTUALS_English.csv
3,During the routine changeover of BOP Transform...,Process Safety,Loss of EDL & Pi Servers/Connection,English,RW_ACTUALS_English.csv
4,Trant dumper driver collided with CDC exit rea...,Process Safety,Damage to CDC Vehicle Exit Gate Reader,English,RW_ACTUALS_English.csv


In [3]:
# =============================================================================
# MODEL CONFIGURATION - Both Flan-T5-Large and Flan-T5-XL
# =============================================================================

# Models to compare
MODELS_TO_COMPARE = [
    "google/flan-t5-large",  # ~780M parameters
    "google/flan-t5-xl"      # ~3B parameters
]

# Dictionary to store results for each model
all_model_results = {}
all_model_metrics = {}

print("[INFO] Models to compare:")
for m in MODELS_TO_COMPARE:
    print(f"  - {m}")
print(f"\n[INFO] Using device: {device}")

[INFO] Models to compare:
  - google/flan-t5-large
  - google/flan-t5-xl

[INFO] Using device: cuda


In [4]:
# =============================================================================
# FEW-SHOT EXAMPLES BY LANGUAGE (ACTUAL EXAMPLES FROM DATA)
# =============================================================================

# Process Safety and Non-Process Safety Examples - ACTUAL examples provided
FEW_SHOT_EXAMPLES = {
    'English': {
        'process_safety': [
            {
                'title': "GT Trip from PRS ESV's closing",
                'description': "At 09.49:33.913 GT Tripped due to humming event caused by low gas pressure from the PRS. Low gas pressure recorded at the PRS due to ESD system operation closing the inlet ESD valve XV4002 and pressure reduction skid valve XV4480. Scada system alarms showed ESD system operated due to PRS common hand switch operated, (PRS\\ESD\\UA4002) and ESD trip from CDC control room (PRS\\ESD\\HS4002C) receipt of this alarm would indicate that the operator himself had activated the PRS trip by pushing the emergency trip button in the plant MCR This cannot of been the cause of the PRS trip due to the operator not pressing the push button in the MCR, this was witnessed by gas operations as they were present in the control room when the ESD system tripped. whilst the gas pressure supply to CDC did fall the cause of the unit trip was from combustion instability (humming). it is recognised that the low gas pressure would have tripped the GT by activation of the low gas pressure trip signals however on this occasion 'Humming' occurred operating the MAX 3 protection trip before the low gas pressure trip signals were received. The load at the time of the trip was 414MWgen (full load in 'OTC operation).",
                'label': 'Process Safety'
            },
            {
                'title': "Unit 4 South East 19m level HRSG casing split, exhaust gasses at extreme temperatures escaping into the HRSG building.",
                'description': "During Production plant checks it was noticed that the section of the HRSG South East walkway at the 19m level in the region of the LP circ pumps was extremely hot, upon further investigation a split was identified within the HRSG casing resulting in the lagging being blown out and exhaust gasses at extreme high temperatures leaking into the area. SAP notification 21495373 refers. As a result of the leak, instrument cables supplying the LP circ pump differential pressure transmitters have been damaged causing the pumps to trip. A SIM has been raised to prevent the pumps from tripping and a unit run back occurring. SIM 21495433 refers.",
                'label': 'Process Safety'
            },
            {
                'title': "U6 PLST Feed Pump Trip On Forced Changeover",
                'description': "Coming on to shift in the morning it was noted that U6 BFP No.2 had developed a discharge v/v gland leak, the unit was due to 2shift overnight and it was decided the safest way to carry out a forced changeover was offload overnight. Around 15:00 we had a phone call from UGC (energy trading) to advise that the unit would be running through overnight. With this new change in commercial running and the fact that the leak was deemed to be deteriorating, the decision was made to try a forced changeover of the feed pumps while the unit was onload but in a steady state. at around 16:10 the changeover was initiated placing BFP No. 2 in service, at which point both feed pumps tripped and the unit PLST'd.",
                'label': 'Process Safety'
            },
            {
                'title': "Oil spill to surface water due to a tipped over IBC with oil and the wash away due to a process water leakage.",
                'description': "While moving an IBC (mounted on a chassis with wheels) on the forks of a forklift by a contractor employee, the IBC and chassis fell over while putting it on the ground. This allegedly happened because the tyres of the chassis of the IBC were too slack, making the IBC and chassis unstable. After the IBC and chassis toppled over, +/- 300L of oil leaked from the IBC via an unsealed opening at the top of the IBC and ended up on the tiled floor. After the incident had occurred and was made safe, the contractor company resumed its work. While moving a pallet with material, a contractor employee hit and damaged a live process water pipe (7.5 bar). Due to the large amount of water released from the process water pipe, the oil spill drifted away, partly ending up in a rainwater drainage next to the hall. This drainage discharges into the Europa harbor via an underground sewer system.",
                'label': 'Process Safety'
            }
        ],
        'non_process_safety': [
            {
                'title': "Paint spillage Tech Centre",
                'description': "Paint was being removed from the COSHH cabinet in the Integral engineer area in the rig bay roof, to be relocated to the COSHH cabinet in the shed. We had made the decision to do this, following the issuance of the FRA and comments made concerning the rig bay roof. Office Essentials loaded the paints onto a rolling cage and took them to the goods lift, where they were moved to the ground floor and out of the external door into the carpark. As they moved the cage down the slope into the car park, the door came undone and a tin of paint fell onto the floor, the lid came off and a small amount of emulsion paint spilled onto the tarmac surface. This was highlighted to David Gray and Cassie Hall who cleaned up the spill with absorbent pads and blue roll from the spill kit. There was a thin residue left on the floor that could not be wiped up further with pads / wipes. David Gray decided to wash down the area with water to dilute what was left of the paint, after no more could be removed by the absorbent pads.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "Operative caught in face whilst trying to thread a steel rope around a crane fly jib pully",
                'description': "Whilst rigging the crane fly jib the wire rope deflected off a fly jib cleat, this struck the IP above his left eye knocking off his safety glasses causing a small cut and slight swelling to his eyebrow and an abrasion to his cheek.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "Cut finger while repairing inlet bridge duct",
                'description': "While carrying out internal repairs to U4 inlet bridge duct IP cut his finger on a piece of metal. The individual was wearing all of the correct PPE while carrying out the repair but as he was brushing the dust from the area a piece of metal cut through his glove and cut his finger. First aid was administered at site and employee returned to work the next day.",
                'label': 'Non-Process Safety'
            }
        ]
    },
    'German': {
        'process_safety': [
            {
                'title': "Stations Not Aus durch Arbeiten am Netzwerk",
                'description': "Aufgrund von Netzwerkstörungen wurde ein defekter Scalance ( Netzwerkswitch ) im Automationsnetzwerk ausgetauscht. Nach Demontage des defekten Gerätes wurde ein Ersatzgerät eingebaut. Nach Anschluss des Ersatzgerätes wurde eine Notabschaltung der Speicherstation initiiert. Alle Anlagenteile wurden in den sicheren Zustand versetzt.",
                'label': 'Process Safety'
            },
            {
                'title': "Leitfähigkeitsanstieg im Deionattank 1&2 durch Eintrag von Brunnenwasser",
                'description': "In der Nacht vom 05.01.2024 auf den 06.01.2024 bemerkte der diensthabende Schichtleiter einen Anstieg der Leitfähigkeit des Deionats nach Deionatpumpen, (Messtelle GCA40FQ001). Um eine Fehlmessung auszuschließen, wurde am Morgen des 06.01.2024 eine Kontrollmessung durch das Kraftwerkslabor durchgeführt, welche die erhöhte Leitfähigkeit bestätigte. Die anschließende Ursachenfindung ergab, dass durch eine nicht korrekt gestellte Armatur (Brunnenwasser zur Temperaturregelung der Oxiluft) in der REA, Brunnenwasser in das Deionatsystem gelangte. Aufgrund des höheren Systemdrucks im Brunnenwassersystem konnte über die normalisierte Deionatleitung rückwärts Brunnenwasser in die Deionattanks 1&2 gelangen.",
                'label': 'Process Safety'
            },
            {
                'title': "Kesselschaden mit Leckage nach außen",
                'description': "Es wurde ein Wasseraustritt (leichtes Tropfen) aus der Isolierung des Brennerkastens von Brenner 3.3 am Kessel des Block 1 festgestellt. Der Block war nicht in Betrieb und der Kessel war kalt. Aufgrund der günstigen Marktlage wurde eine NV eingestellt und die Reparatur in der Folgewoche durchgeführt. Die Analyse nach Abisolieren eruierte den Riss als unkritisch, nach Reparatur keine weiteren Maßnahmen erforderlich.",
                'label': 'Process Safety'
            }
        ],
        'non_process_safety': [
            {
                'title': "Erste-Hilfe - Finger geklemmt an Schraubstock Schülerpraktikant",
                'description': "Beim Einspannen eines Werkstückes in den Schraubstock durch den Schülerpraktikant ist der Drehgriff (Griffstück/ Kurbelstab) auf den kleinen Finger nach unten gefallen und hat diesen leicht gezwickt.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "EDE - Ausbildungswerkstatt 32A CEE Steckdose",
                'description': "32A CEE Steckvorrichtung - Die Befestigung ist defekt/locker",
                'label': 'Non-Process Safety'
            },
            {
                'title': "WA1/BRI - OBK - Undichtigkeit Hydraulikschlauch",
                'description': "Undichtigkeit Hydraulikschlauchleitung an Bohrgerät auf der Mauerkrone der südlichen Ringmauer Oberbecken PSW Waldeck 1. Ergriffene Schutzmaßnahmen (Erstellung Wannenplane) hat gegriffen.",
                'label': 'Non-Process Safety'
            }
        ]
    },
    'Dutch': {
        'process_safety': [
            {
                'title': "Olie spill naar oppervlaktewater door omgevallen kubel met olie en het wegspoelen door proceswaterlekkage.",
                'description': "Tijdens het verplaatsen van een kubel (gemonteerd op een onderstel met wielen) op de lepels van een heftruck door een contractormedewerker is de kubel met onderstel tijdens het neerzetten omgevallen. Dit heeft naar zeggen kunnen gebeuren doordat de banden van het onderstel van de kubel te slap waren waardoor deze onstabiel was. Nadat de kubel met onderstel was omgevallen is er +/- 300L olie uit de kubel gelekt via een niet afgesloten opening aan de bovenkant van de kubel en terecht gekomen op de tegelvloer. Nadat het incident had plaatsgevonden heeft een contractormedewerker een in bedrijf zijnde proceswaterleiding (7,5 bar) geraakt en beschadigd. Door de grote hoeveelheid water die vrij kwam uit de proceswaterleiding is de olie spill weggedreven en daarbij deels in een HWA goot naast de hal terecht gekomen. Deze HWA afvoer komt via ondergrondse riolering direct uit in de Europa haven.",
                'label': 'Process Safety'
            },
            {
                'title': "Mogelijk twee soorten glycol in koelwatersysteem Roca 1 en Roca 2.",
                'description': "In de eerste helft van 2022 is glycol Vidol Glycocare vervangen door glycol VIB Glysantin G40. Er is door mensen beweerd dat deze twee soorten prima bij elkaar kunnen. Nu, eind 2023 treden er oa in de leidingen van de HVAC machines verstoppingen op door blauwe kristalachtige deeltjes en deeltjes die op ijzer en steentjes lijken. Misschien kunnen de twee soorten glycol toch niet goed bij elkaar. Volgens mensen die bij dit project betrokken zijn geweest is de glycol tank leeggemaakt en schoongemaakt maar het koelwatersysteem zelf niet.",
                'label': 'Process Safety'
            }
        ],
        'non_process_safety': [
            {
                'title': "ontsteking kolendamp tijdens broeibestrijding A2 veld",
                'description': "Tijdens het bestrijden van broei op kolenveld A2 is een deel van de warme dampen ontstoken door de hoge temperatuur en heeft een kleine plof veroorzaakt, waarna de hele hoek van het veld kortstondig brandde. Gelukkig stonden we hier op moment van ontsteken ca 2 meter vandaan, Als we op dat moment met de schep de broei aan het bestrijden waren geweest hadden hier mogelijk brandwonden opgelopen kunnen worden. Door de huidige situaties rondom de kolenvelden (langer liggen van de kolen door regelmatige stilstand) zal de situatie met broei op de velden zich de komende jaren waarschijnlijk vaker voor doen.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "Incident met een ingehuurde aannemer voor slijp werkzaamheden aan het T-12 materiaal.",
                'description': "Donderdag 25 augustus 2022 is er gewerkt aan het T12 materiaal door mensen van Ergenc. Zij waren delen aan het los slijpen. Hierbij heeft zich rond 09:48 een incident voor gedaan waarbij een medewerkers gevallen is. Hij leunde op een gedeelte T-12 materiaal dat na het slijpen naar beneden gevallen is. Het incident dat zich heeft voor gedaan betreft een ongeval zonder verzuim.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "Onboarding externe verloopt niet (meer) correct.",
                'description': "Het onboardingsproces is dermate gewijzigd waardoor externe geen verplichte trainingen krijgen toegewezen in het LMS Archipel / correcte toegangsrechten / kleding / locker etc. Het betreft hierbij externe medewerkers welke een gelijke functie uitvoeren als een Uniper Medewerker. De verplichte trainingen aan de hand van de functie matrix worden toegewezen welke bij die functie behoort. Er worden nu geen trainingen toegewezen.",
                'label': 'Non-Process Safety'
            }
        ]
    },
    'Swedish': {
        'process_safety': [
            {
                'title': "HVT - G11 skorsten: Asfaltsskada vid kranlyft",
                'description': "HVT - G11 skorsten: Asfaltsskada vid kranlyft",
                'label': 'Process Safety'
            }
        ],
        'non_process_safety': [
            {
                'title': "Oskarshamn 0 - OLY - Person stukade foten i trappa.",
                'description': "Oskarshamn 0 - OLY - Person stukade foten i trappa.",
                'label': 'Non-Process Safety'
            },
            {
                'title': "Oskarshamn 0 - TIL - En kollega till mig hade igår 16:e januari oturen att inte se vad väggrenen var på grund av tillfälligt dålig sikt.",
                'description': "Oskarshamn 0 - TIL - En kollega till mig hade igår 16:e januari oturen att inte se vad väggrenen var på grund av tillfälligt dålig sikt.",
                'label': 'Non-Process Safety'
            }
        ]
    }
}

print("[OK] Few-shot examples defined using ACTUAL examples from data")
print(f"  English: {len(FEW_SHOT_EXAMPLES['English']['process_safety'])} PS, {len(FEW_SHOT_EXAMPLES['English']['non_process_safety'])} NPS")
print(f"  German: {len(FEW_SHOT_EXAMPLES['German']['process_safety'])} PS, {len(FEW_SHOT_EXAMPLES['German']['non_process_safety'])} NPS")
print(f"  Dutch: {len(FEW_SHOT_EXAMPLES['Dutch']['process_safety'])} PS, {len(FEW_SHOT_EXAMPLES['Dutch']['non_process_safety'])} NPS")
print(f"  Swedish: {len(FEW_SHOT_EXAMPLES['Swedish']['process_safety'])} PS, {len(FEW_SHOT_EXAMPLES['Swedish']['non_process_safety'])} NPS")

[OK] Few-shot examples defined using ACTUAL examples from data
  English: 4 PS, 3 NPS
  German: 3 PS, 3 NPS
  Dutch: 2 PS, 3 NPS
  Swedish: 1 PS, 2 NPS


In [5]:
# =============================================================================
# PROCESS SAFETY DEFINITION AND PROMPT TEMPLATES
# =============================================================================

PROCESS_SAFETY_DEFINITION = """
A Process Safety Incident is ANY incident involving:
1. Equipment failures, trips, or malfunctions (GT trip, pump trip, boiler trip, valve failure)
2. Leaks, spills, or releases of process materials (oil, gas, water, chemicals, steam from equipment)
3. Pressure deviations (high/low pressure events)
4. Emergency shutdowns (ESD, Not-Aus, noodstop, emergency stop)
5. Fire, explosion, or rupture
6. Loss of containment from process equipment
7. Contamination events (wrong substance in system)
8. Equipment damage (cracks, splits, corrosion)
9. Unplanned unit outages or trips
10. Any incident with potential for harm to people, environment, or equipment FROM PROCESS SYSTEMS

A Non-Process Safety Incident includes:
- Personal injuries (cuts, falls, sprains) NOT caused by process equipment failure
- Minor spills of non-process materials (paint, office supplies)
- Administrative matters, training, onboarding issues
- Occupational safety incidents (slips, trips, falls, manual handling injuries)
- Electrical socket/outlet issues in workshops (not process systems)
- Vehicle/driving incidents
- Near misses involving people but NOT process equipment
"""

def build_few_shot_prompt(title, description, language='English'):
    """
    Build a few-shot classification prompt with examples from both classes.
    """
    examples = FEW_SHOT_EXAMPLES.get(language, FEW_SHOT_EXAMPLES['English'])
    
    # Build examples section with BOTH classes
    examples_text = ""
    
    # Add Process Safety examples
    for ex in examples['process_safety'][:2]:
        examples_text += f"""
Example:
Title: {ex['title']}
Description: {ex['description'][:500]}...
Classification: Process Safety
"""
    
    # Add Non-Process Safety examples
    for ex in examples['non_process_safety'][:2]:
        examples_text += f"""
Example:
Title: {ex['title']}
Description: {ex['description'][:500]}...
Classification: Non-Process Safety
"""
    
    prompt = f"""You are an expert industrial safety analyst. Classify incidents as "Process Safety" or "Non-Process Safety".

{PROCESS_SAFETY_DEFINITION}

{examples_text}

Now classify this incident:
Title: {title}
Description: {description}

KEY DISTINCTIONS:
- Process Safety: Equipment failures, leaks from process systems, emergency shutdowns, fires, explosions
- Non-Process Safety: Personal injuries (cuts, falls), minor non-process spills, administrative issues

Classification (answer ONLY "Process Safety" or "Non-Process Safety"):"""
    
    return prompt

# Display a sample prompt
sample_row = master_df_7.iloc[0]
sample_prompt = build_few_shot_prompt(
    sample_row['TITLE'], 
    sample_row['CASE_DESCRIPTION'],
    sample_row.get('LANGUAGE', 'English')
)
print("[INFO] Sample prompt structure:")
print("=" * 80)
print(sample_prompt[:2500] + "..." if len(sample_prompt) > 2500 else sample_prompt)
print("=" * 80)

[INFO] Sample prompt structure:
You are an expert industrial safety analyst. Classify incidents as "Process Safety" or "Non-Process Safety".


A Process Safety Incident is ANY incident involving:
1. Equipment failures, trips, or malfunctions (GT trip, pump trip, boiler trip, valve failure)
2. Leaks, spills, or releases of process materials (oil, gas, water, chemicals, steam from equipment)
3. Pressure deviations (high/low pressure events)
4. Emergency shutdowns (ESD, Not-Aus, noodstop, emergency stop)
5. Fire, explosion, or rupture
6. Loss of containment from process equipment
7. Contamination events (wrong substance in system)
8. Equipment damage (cracks, splits, corrosion)
9. Unplanned unit outages or trips
10. Any incident with potential for harm to people, environment, or equipment FROM PROCESS SYSTEMS

A Non-Process Safety Incident includes:
- Personal injuries (cuts, falls, sprains) NOT caused by process equipment failure
- Minor spills of non-process materials (paint, office sup

In [6]:
# =============================================================================
# COMPREHENSIVE PROCESS SAFETY KEYWORDS (CCPS-Based, Multilingual)
# =============================================================================

# Organized by category based on CCPS Process Safety Metrics guidelines

PS_KEYWORDS = {
    # =========================================================================
    # ENGLISH KEYWORDS
    # =========================================================================
    'English': [
        # 1) Loss of Containment (LoC) and Release Terminology
        # General release terms
        'leak', 'leakage', 'leaking', 'seepage', 'weep', 'dripping', 'spray', 'jet', 'discharge',
        'release', 'escape', 'venting', 'blowdown', 'rupture', 'burst', 'split', 'cracked', 'fracture',
        'failure', 'breach', 'perforation', 'pinhole', 'blowout', 'spill', 'spillage', 'overflow',
        'overfill', 'tank overfill', 'product release', 'chemical release', 'gas release', 'vapor release',
        
        # Common LoC locations
        'pipe', 'piping', 'pipeline', 'line', 'header', 'manifold', 'flange', 'gasket', 'joint',
        'coupling', 'fitting', 'connector', 'valve', 'actuator', 'valve passing', 'valve stuck open',
        'valve stuck closed', 'valve failure', 'seal', 'mechanical seal', 'packing', 'o-ring',
        'seal failure', 'weld', 'weld failure', 'nozzle', 'drain', 'vent', 'sample point',
        'hose', 'flexible hose', 'hose rupture', 'tank', 'vessel', 'pressure vessel', 'drum',
        'reactor', 'column', 'exchanger', 'boiler', 'hrsg', 'furnace', 'heater',
        
        # Containment / secondary containment
        'bund', 'sump', 'containment', 'secondary containment', 'catch basin',
        
        # 2) Fire, Explosion, and Ignition Indicators
        # Fire / combustion
        'fire', 'flame', 'burning', 'smoldering', 'smoke', 'soot', 'charring', 'scorch', 'flash',
        'flare-up', 'ignition', 'ignited', 'spark', 'arcing', 'hot surface', 'hot spot', 'glowing',
        'embers', 'flash fire', 'pool fire', 'jet fire',
        
        # Explosion / deflagration
        'explosion', 'explosive', 'blast', 'detonation', 'deflagration', 'overpressure event',
        'pressure wave', 'blast damage',
        
        # Flammable atmosphere
        'flammable gas', 'flammable vapor', 'lel', 'uel', '%lel', 'gas cloud', 'vapor cloud',
        'combustible', 'explosive mixture', 'hydrocarbon vapor',
        
        # 3) Toxic / Asphyxiant Releases
        'toxic gas', 'toxic vapor', 'poisoning', 'inhalation hazard', 'h2s', 'hydrogen sulfide',
        'ammonia', 'nh3', 'chlorine', 'cl2', 'phosgene', 'so2', 'nox', 'co', 'carbon monoxide',
        'benzene', 'voc', 'solvent vapor', 'oxygen deficiency', 'o2 low', 'nitrogen purge',
        'inerting', 'co2 release', 'asphyxiant',
        
        # 4) High-Energy / High-Hazard Process Conditions
        # Pressure excursions
        'overpressure', 'high pressure', 'pressure spike', 'pressure surge', 'pressure excursion',
        'psv lift', 'safety valve lift', 'relief valve lifted', 'pressure relief', 'vent to flare',
        'rupture disk', 'bursting disc', 'flare', 'vent stack', 'vacuum collapse', 'low pressure',
        'implosion',
        
        # Temperature excursions
        'overtemperature', 'overheating', 'thermal runaway', 'hot oil', 'high temperature excursion',
        'furnace trip', 'heater tube leak', 'refractory failure', 'cryogenic leak',
        
        # Flow / level excursions
        'runaway flow', 'reverse flow', 'backflow', 'siphoning', 'high level alarm', 'low level alarm',
        'tank rollover', 'flooding',
        
        # 5) Mechanical Integrity / Material Degradation
        'corrosion', 'erosion', 'thinning', 'wall loss', 'pitting', 'crack', 'fatigue',
        'brittle fracture', 'creep', 'stress corrosion cracking', 'scc', 'mechanical failure',
        'structural failure', 'integrity failure', 'catastrophic failure', 'deformation', 'bulging',
        'bowing', 'buckling', 'collapsed', 'loose bolts', 'bolt failure', 'stud failure',
        
        # 6) Rotating Equipment and Power-Plant Signals
        'turbine trip', 'compressor trip', 'pump trip', 'gt trip', 'overspeed', 'runaway',
        'high vibration', 'bearing failure', 'lube oil leak', 'seal oil leak', 'steam leak',
        'high-pressure steam leak', 'condensate leak', 'hrsg leak', 'boiler tube leak',
        'tube rupture', 'economizer leak', 'superheater leak', 'gas turbine exhaust leak',
        'casing leak', 'hot gas leak', 'casing split',
        
        # 7) Instrumentation, Alarms, Interlocks, Safety Systems
        'alarm', 'high-high alarm', 'hh alarm', 'low-low alarm', 'll alarm', 'trip', 'shutdown',
        'plant trip', 'unit trip', 'interlock', 'permissive', 'bypassed interlock', 'inhibited alarm',
        'override', 'suppression', 'sis', 'safety instrumented system', 'sif', 'esd',
        'emergency shutdown', 'trip logic', 'gas detector', 'gas detection', 'h2s detector',
        'lel alarm', 'fire & gas', 'f&g', 'relief system', 'flare system', 'blowdown valve',
        'vent valve', 'psv stuck', 'relief valve failed', 'failed to lift', 'chatter',
        'leakage through psv',
        
        # 8) Electrical / Static / Lightning
        'electrical arc', 'short circuit', 'static discharge', 'electrostatic',
        'bonding failure', 'grounding failure', 'lightning strike',
        
        # 9) Near-Miss and Escalation Language
        'near miss', 'potential', 'could have', 'narrowly avoided', 'prevented', 'contained',
        'mitigated', 'escalation', 'major incident potential', 'high potential', 'high consequence',
        'barrier', 'protection', 'safeguard', 'layer of protection', 'lopa', 'abnormal',
        'deviation', 'excursion', 'out of specification', 'loss of control', 'emergency response',
        'evacuation',
        
        # 10) Substance/Material Keywords
        'hydrocarbon', 'fuel gas', 'natural gas', 'lpg', 'lng', 'propane', 'butane', 'gasoline',
        'kerosene', 'diesel', 'hydrogen', 'ethylene', 'acetylene', 'methane', 'solvent', 'methanol',
        'ethanol', 'toluene', 'xylene', 'flammable', 'combustible', 'volatile', 'pressurized',
        'high temperature', 'toxic', 'corrosive', 'hazardous material', 'hazmat',
    ],
    
    # =========================================================================
    # GERMAN KEYWORDS
    # =========================================================================
    'German': [
        # 1) Leckage und Freisetzung
        'leck', 'leckage', 'undichtigkeit', 'undicht', 'tropfen', 'tropfleck', 'austritt',
        'freisetzung', 'entweichen', 'abblasen', 'entspannen', 'bruch', 'riss', 'bersten',
        'aufplatzen', 'versagen', 'durchbruch', 'perforation', 'lochfraß', 'blowout',
        'verschüttung', 'überlauf', 'überfüllung', 'tanküberlauf', 'produktfreisetzung',
        'chemikalienfreisetzung', 'gasfreisetzung', 'dampffreisetzung',
        
        # LoC-Orte
        'rohr', 'rohrleitung', 'pipeline', 'leitung', 'sammelleitung', 'verteiler',
        'flansch', 'dichtung', 'verbindung', 'kupplung', 'fitting', 'stecker',
        'ventil', 'armatur', 'stellantrieb', 'ventil klemmt', 'ventilversagen',
        'dichtung', 'gleitringdichtung', 'packung', 'o-ring', 'dichtungsversagen',
        'schweißnaht', 'schweißnahtversagen', 'stutzen', 'ablass', 'entlüftung', 'probenahmestelle',
        'schlauch', 'flexibler schlauch', 'schlauchbruch', 'tank', 'behälter', 'druckbehälter',
        'trommel', 'reaktor', 'kolonne', 'wärmetauscher', 'kessel', 'hrsg', 'ofen', 'erhitzer',
        
        # Auffangwanne
        'auffangwanne', 'sumpf', 'rückhaltung', 'sekundäre rückhaltung',
        
        # 2) Feuer, Explosion, Zündung
        'feuer', 'brand', 'flamme', 'brennen', 'schwelen', 'rauch', 'ruß', 'verkohlung',
        'versengen', 'blitz', 'aufflackern', 'zündung', 'gezündet', 'funken', 'lichtbogen',
        'heiße oberfläche', 'hot spot', 'glühen', 'glut', 'blitzbrand', 'lachenbrand', 'strahlbrand',
        'explosion', 'explosiv', 'detonation', 'verpuffung', 'druckwelle', 'explosionsschaden',
        'brennbares gas', 'brennbarer dampf', 'ueg', 'oeg', 'gaswolke', 'dampfwolke',
        'brennbar', 'explosionsfähiges gemisch', 'kohlenwasserstoffdampf',
        
        # ) Toxische Freisetzungen
        'giftgas', 'giftiger dampf', 'vergiftung', 'einatemgefahr', 'h2s', 'schwefelwasserstoff',
        'ammoniak', 'nh3', 'chlor', 'cl2', 'phosgen', 'so2', 'nox', 'co', 'kohlenmonoxid',
        'benzol', 'voc', 'lösungsmitteldampf', 'sauerstoffmangel', 'o2 niedrig', 'stickstoffspülung',
        'inertisierung', 'co2-freisetzung', 'erstickungsgefahr',
        
        # 4) Druck- und Temperaturabweichungen
        'überdruck', 'hochdruck', 'druckspitze', 'druckstoß', 'druckabweichung',
        'sicherheitsventil angehoben', 'druckentlastung', 'abblasen zur fackel',
        'berstscheibe', 'fackel', 'entlüftungsleitung', 'vakuumkollaps', 'unterdruck', 'implosion',
        'übertemperatur', 'überhitzung', 'thermisches durchgehen', 'heißöl', 'hochtemperaturabweichung',
        'ofenausfall', 'heizrohrleck', 'feuerfestversagen', 'kryogenes leck',
        'durchgehender durchfluss', 'rückfluss', 'rücksaugen', 'hochstandalarm', 'tiefstandalarm',
        'tankrollover', 'überflutung',
        
        # 5) Mechanische Integrität
        'korrosion', 'erosion', 'ausdünnung', 'wandverlust', 'lochfraß', 'riss', 'ermüdung',
        'sprödbruch', 'kriechen', 'spannungsrisskorrosion', 'srk', 'mechanisches versagen',
        'strukturversagen', 'integritätsversagen', 'katastrophales versagen', 'verformung',
        'ausbeulung', 'durchbiegung', 'knicken', 'eingestürzt', 'lose schrauben', 'schraubenversagen',
        
        # 6) Rotierende Ausrüstung
        'turbinenausfall', 'kompressorausfall', 'pumpenausfall', 'gt-trip', 'überdrehzahl',
        'durchgehen', 'hohe vibration', 'lagerversagen', 'schmierölleck', 'dichtölleck',
        'dampfleck', 'hochdruckdampfleck', 'kondensatleck', 'hrsg-leck', 'kesselrohrleck',
        'rohrbruch', 'economizer-leck', 'überhitzer-leck', 'gasturbinenabgasleck',
        'gehäuseleck', 'heißgasleck', 'gehäuseriss',
        
        # 7) Alarme und Sicherheitssysteme
        'alarm', 'hochhoch-alarm', 'hh-alarm', 'tieftief-alarm', 'll-alarm', 'trip', 'abschaltung',
        'anlagenausfall', 'blockausfall', 'verriegelung', 'freigabe', 'überbrückte verriegelung',
        'unterdrückter alarm', 'übersteuerung', 'sis', 'sicherheitsinstrumentiertes system',
        'sif', 'not-aus', 'notabschaltung', 'schnellabschaltung', 'gasdetektor', 'gaserkennung',
        'h2s-detektor', 'ueg-alarm', 'feuer & gas', 'f&g', 'entlastungssystem', 'fackelsystem',
        'abblaseventil', 'entlüftungsventil', 'psv klemmt', 'überdruckventil versagt',
        
        # 8) Elektrisch / Statisch
        'elektrischer lichtbogen', 'kurzschluss', 'elektrostatische entladung',
        'erdungsversagen', 'blitzeinschlag',
        
        # 9) Beinaheunfall und Eskalation
        'beinaheunfall', 'potenzial', 'hätte können', 'knapp vermieden', 'verhindert', 'eingedämmt',
        'gemildert', 'eskalation', 'hohes unfallpotenzial', 'hohe konsequenz', 'barriere',
        'schutz', 'schutzschicht', 'lopa', 'abnormal', 'abweichung', 'außerhalb der spezifikation',
        'kontrollverlust', 'notfallreaktion', 'evakuierung',
        
        # 10) Stoffbezeichnungen
        'kohlenwasserstoff', 'brenngas', 'erdgas', 'lpg', 'lng', 'propan', 'butan', 'benzin',
        'kerosin', 'diesel', 'wasserstoff', 'ethylen', 'acetylen', 'methan', 'lösungsmittel',
        'methanol', 'ethanol', 'toluol', 'xylol', 'brennbar', 'entzündlich', 'flüchtig',
        'unter druck', 'hochtemperatur', 'giftig', 'ätzend', 'gefahrstoff',
    ],
    
    # =========================================================================
    # DUTCH KEYWORDS
    # =========================================================================
    'Dutch': [
        # 1) Lekkage en Vrijgave
        'lek', 'lekkage', 'lekken', 'sijpelen', 'druppelen', 'spray', 'straal', 'lozing',
        'vrijgave', 'ontsnappen', 'ontluchten', 'afblazen', 'breuk', 'barsten', 'scheuren',
        'gescheurd', 'falen', 'doorbraak', 'perforatie', 'putcorrosie', 'blowout',
        'morsen', 'morsing', 'overloop', 'overvulling', 'tankoverloop', 'productvrijgave',
        'chemische vrijgave', 'gasvrijgave', 'dampvrijgave',
        
        # LoC-locaties
        'pijp', 'leiding', 'pijpleiding', 'lijn', 'verzamelleiding', 'verdeler',
        'flens', 'pakking', 'verbinding', 'koppeling', 'fitting', 'connector',
        'klep', 'afsluiter', 'actuator', 'klep lekt door', 'klep vastzittend', 'klepfalen',
        'afdichting', 'mechanische afdichting', 'pakking', 'o-ring', 'afdichtingsfalen',
        'las', 'lasfalen', 'spuitstuk', 'aftap', 'ontluchting', 'monsterpunt',
        'slang', 'flexibele slang', 'slangbreuk', 'tank', 'vat', 'drukvat', 'drum',
        'reactor', 'kolom', 'wisselaar', 'ketel', 'hrsg', 'oven', 'verwarmer',
        
        # Opvang
        'inkuiping', 'put', 'opvang', 'secundaire opvang', 'opvangbak',
        
        # 2) Brand, Explosie, Ontsteking
        'brand', 'vuur', 'vlam', 'branden', 'smeulen', 'rook', 'roet', 'verkolen',
        'schroeien', 'flits', 'opvlammen', 'ontsteking', 'ontstoken', 'vonk', 'vlamboog',
        'heet oppervlak', 'hot spot', 'gloeien', 'sintels', 'flitsbrand', 'plasbrand', 'straalbrand',
        'explosie', 'explosief', 'detonatie', 'deflagratie', 'overdrukgebeurtenis',
        'drukgolf', 'explosie schade',
        'brandbaar gas', 'brandbare damp', 'lel', 'uel', '%lel', 'gaswolk', 'dampwolk',
        'brandbaar', 'explosief mengsel', 'koolwaterstofdamp',
        
        # 3) Toxische Vrijgaves
        'giftig gas', 'giftige damp', 'vergiftiging', 'inhalatiegevaar', 'h2s', 'waterstofsulfide',
        'ammoniak', 'nh3', 'chloor', 'cl2', 'fosgeen', 'so2', 'nox', 'co', 'koolmonoxide',
        'benzeen', 'voc', 'oplosmiddeldamp', 'zuurstoftekort', 'o2 laag', 'stikstof spoelen',
        'inertiseren', 'co2-vrijgave', 'verstikkingsgevaar',
        
        # 4) Druk- en Temperatuurafwijkingen
        'overdruk', 'hoge druk', 'drukpiek', 'drukstoot', 'drukafwijking',
        'veiligheidsklep gelicht', 'drukontlasting', 'afblazen naar fakkel',
        'breekplaat', 'fakkel', 'ontluchtingsleiding', 'vacuüm ineenstorting', 'onderdruk', 'implosie',
        'overtemperatuur', 'oververhitting', 'thermische doorslaag', 'hete olie', 'hoge temperatuurafwijking',
        'ovenuitval', 'verwarmingsbuislek', 'vuurvast falen', 'cryogeen lek',
        'doorslaan stroom', 'terugstroom', 'terugslaggen', 'sifon', 'hoog niveau alarm',
        'laag niveau alarm', 'tank rollover', 'overstroming',
        
        # 5) Mechanische Integriteit
        'corrosie', 'erosie', 'wandverdunning', 'wandverlies', 'putcorrosie', 'scheur', 'vermoeiing',
        'brosse breuk', 'kruip', 'spanningscorrosiescheuren', 'scc', 'mechanisch falen',
        'constructief falen', 'integriteitsfalen', 'catastrofaal falen', 'vervorming',
        'uitstulping', 'doorbuiging', 'knikken', 'ingestort', 'losse bouten', 'boutfalen',
        
        # 6) Roterende Apparatuur
        'turbine trip', 'compressor trip', 'pomp trip', 'gt trip', 'te hoge snelheid',
        'op hol slaan', 'hoge vibratie', 'lagerfalen', 'smeerolie lek', 'afdichtingsolie lek',
        'stoomlek', 'hogedruk stoomlek', 'condensaatlek', 'hrsg lek', 'ketelbuislek',
        'buisbreuk', 'economizer lek', 'oververhitter lek', 'gasturbine uitlaatlek',
        'behuizingslek', 'heet gaslek', 'behuizingsscheur',
        
        # 7) Alarmen en Veiligheidssystemen
        'alarm', 'hoog-hoog alarm', 'hh alarm', 'laag-laag alarm', 'll alarm', 'trip', 'uitschakeling',
        'installatie trip', 'unit trip', 'interlock', 'vrijgave', 'overbrugde interlock',
        'onderdrukt alarm', 'override', 'sis', 'veiligheidsinstrumentsysteem', 'sif',
        'noodstop', 'nooduitschakeling', 'trip logica', 'gasdetector', 'gasdetectie',
        'h2s detector', 'lel alarm', 'brand & gas', 'f&g', 'ontlastingssysteem', 'fakkelsysteem',
        'afblaasklep', 'ontluchtingsklep', 'psv vastzittend', 'overdrukklep gefaald',
        
        # 8) Elektrisch / Statisch
        'elektrische boog', 'kortsluiting', 'statische ontlading', 'elektrostatisch',
        'aardingsfalen', 'blikseminslag',
        
        # 9) Bijna-Ongeluk en Escalatie
        'bijna ongeluk', 'potentieel', 'had kunnen', 'net vermeden', 'voorkomen', 'ingeperkt',
        'gemitigeerd', 'escalatie', 'hoog incident potentieel', 'hoge consequentie', 'barrière',
        'bescherming', 'beveiligingslaag', 'lopa', 'abnormaal', 'afwijking', 'buiten specificatie',
        'controleverlies', 'noodrespons', 'evacuatie',
        
        # 10) Stofnamen
        'koolwaterstof', 'brandstofgas', 'aardgas', 'lpg', 'lng', 'propaan', 'butaan', 'benzine',
        'kerosine', 'diesel', 'waterstof', 'ethyleen', 'acetyleen', 'methaan', 'oplosmiddel',
        'methanol', 'ethanol', 'tolueen', 'xyleen', 'brandbaar', 'ontvlambaar', 'vluchtig',
        'onder druk', 'hoge temperatuur', 'giftig', 'bijtend', 'gevaarlijke stof',
    ],
    
    # =========================================================================
    # SWEDISH KEYWORDS
    # =========================================================================
    'Swedish': [
        # 1) Läckage och Utsläpp
        'läcka', 'läckage', 'läcker', 'sipprar', 'droppar', 'spray', 'stråle', 'utsläpp',
        'frigörande', 'undkomma', 'ventilera', 'avblåsning', 'brott', 'spricka', 'bristning',
        'sprucken', 'fel', 'genombrott', 'perforering', 'gropfrätning', 'blowout',
        'spill', 'spillning', 'överflöd', 'överfyllning', 'tanköverflöd', 'produktutsläpp',
        'kemikalieutsläpp', 'gasutsläpp', 'ångutsläpp',
        
        # LoC-platser
        'rör', 'rörledning', 'pipeline', 'ledning', 'samlingsledning', 'förgrening',
        'fläns', 'packning', 'fog', 'koppling', 'fitting', 'anslutning',
        'ventil', 'ställdon', 'ventil läcker', 'ventil fastnat', 'ventilfel',
        'tätning', 'mekanisk tätning', 'packbox', 'o-ring', 'tätningsfel',
        'svets', 'svetsfel', 'studs', 'avtappning', 'avluftning', 'provtagningspunkt',
        'slang', 'flexibel slang', 'slangbrott', 'tank', 'kärl', 'tryckkärl', 'fat',
        'reaktor', 'kolonn', 'värmeväxlare', 'panna', 'hrsg', 'ugn', 'värmeanordning',
        
        # Uppsamling
        'invallning', 'sump', 'uppsamling', 'sekundär uppsamling',
        
        # 2) Brand, Explosion, Antändning
        'brand', 'eld', 'låga', 'brinna', 'pyra', 'rök', 'sot', 'förkolning',
        'sveda', 'blixt', 'uppflammande', 'antändning', 'antänd', 'gnista', 'ljusbåge',
        'het yta', 'hot spot', 'glöda', 'glöd', 'blixtbrand', 'poolbrand', 'strålbrand',
        'explosion', 'explosiv', 'detonation', 'deflagration', 'övertryckshändelse',
        'tryckvåg', 'explosionsskada',
        'brännbar gas', 'brännbar ånga', 'lel', 'uel', '%lel', 'gasmoln', 'ångmoln',
        'brännbar', 'explosiv blandning', 'kolvätedånga',
        
        # 3) Toxiska Utsläpp
        'giftig gas', 'giftig ånga', 'förgiftning', 'inandningsfara', 'h2s', 'vätesulfid',
        'ammoniak', 'nh3', 'klor', 'cl2', 'fosgen', 'so2', 'nox', 'co', 'kolmonoxid',
        'bensen', 'voc', 'lösningsmedelsånga', 'syrebrist', 'o2 låg', 'kvävgasspolning',
        'inertisering', 'co2-utsläpp', 'kvävningsrisk',
        
        # 4) Tryck- och Temperaturavvikelser
        'övertryck', 'högt tryck', 'tryckspik', 'tryckstöt', 'tryckavvikelse',
        'säkerhetsventil lyft', 'trycklättnad', 'avblåsning till fackla',
        'sprängbleck', 'fackla', 'avluftningsledning', 'vakuumkollaps', 'undertryck', 'implosion',
        'övertemperatur', 'överhettning', 'termisk rusning', 'het olja', 'hög temperaturavvikelse',
        'ugnsstopp', 'värmarrörsläcka', 'eldfast fel', 'kryogen läcka',
        'skenande flöde', 'bakflöde', 'hävert', 'hög nivå larm', 'låg nivå larm',
        'tankrullning', 'översvämning',
        
        # 5) Mekanisk Integritet
        'korrosion', 'erosion', 'väggförtunning', 'väggförlust', 'gropfrätning', 'spricka', 'utmattning',
        'sprött brott', 'kryp', 'spänningskorrosion', 'scc', 'mekaniskt fel',
        'strukturellt fel', 'integritetsfel', 'katastrofalt fel', 'deformation',
        'utbuktning', 'böjning', 'buckling', 'kollapsad', 'lösa bultar', 'bultfel',
        
        # 6) Roterande Utrustning
        'turbinstopp', 'kompressorstopp', 'pumpstopp', 'gt trip', 'överhastighet',
        'skenande', 'hög vibration', 'lagerfel', 'smörjoljeläcka', 'tätningsoljeläcka',
        'ångläcka', 'högtrycksångläcka', 'kondensatläcka', 'hrsg läcka', 'pannrörsläcka',
        'rörbrist', 'ekonomizerläcka', 'överhettarläcka', 'gasturbinutsläpp',
        'husläcka', 'het gasläcka', 'husspricka',
        
        # 7) Larm och Säkerhetssystem
        'larm', 'hög-hög larm', 'hh larm', 'låg-låg larm', 'll larm', 'trip', 'avstängning',
        'anläggningsstopp', 'enhetsstopp', 'interlock', 'tillstånd', 'förbikopplad interlock',
        'undertryckt larm', 'överstyrning', 'sis', 'säkerhetsinstrumenterat system', 'sif',
        'nödstopp', 'nödavstängning', 'trip logik', 'gasdetektor', 'gasdetektering',
        'h2s detektor', 'lel larm', 'brand & gas', 'f&g', 'avlastningssystem', 'fackelsystem',
        'avblåsningsventil', 'avluftningsventil', 'psv fastnat', 'säkerhetsventil felat',
        
        # 8) Elektriskt / Statiskt
        'elektrisk ljusbåge', 'kortslutning', 'statisk urladdning', 'elektrostatisk',
        'jordningsfel', 'blixtnedslag',
        
        # 9) Tillbud och Eskalering
        'tillbud', 'potentiell', 'kunde ha', 'nätt undvikit', 'förhindrad', 'innesluten',
        'mildrad', 'eskalering', 'hög incidentpotential', 'hög konsekvens', 'barriär',
        'skydd', 'skyddslager', 'lopa', 'onormal', 'avvikelse', 'utanför specifikation',
        'kontrollförlust', 'nödrespons', 'evakuering',
        
        # 10) Ämnesnamn
        'kolväte', 'bränslegas', 'naturgas', 'lpg', 'lng', 'propan', 'butan', 'bensin',
        'fotogen', 'diesel', 'väte', 'etylen', 'acetylen', 'metan', 'lösningsmedel',
        'metanol', 'etanol', 'toluen', 'xylen', 'brännbar', 'antändbar', 'flyktig',
        'trycksatt', 'hög temperatur', 'giftig', 'frätande', 'farligt ämne',
    ]
}

# Flatten all keywords into a single set for fast lookup
ALL_PS_KEYWORDS = set()
for lang_keywords in PS_KEYWORDS.values():
    ALL_PS_KEYWORDS.update(kw.lower() for kw in lang_keywords)

def count_ps_keywords(text, language='English'):
    """Count how many PS keywords are present in the text."""
    text_lower = text.lower()
    # Use language-specific keywords plus all keywords for better coverage
    lang_keywords = set(kw.lower() for kw in PS_KEYWORDS.get(language, PS_KEYWORDS['English']))
    # Combine with all keywords for multilingual documents
    combined_keywords = lang_keywords.union(ALL_PS_KEYWORDS)
    return sum(1 for kw in combined_keywords if kw in text_lower)

def classify_incident_with_model(title, description, tokenizer, model, device, language='English', max_length=1024, max_new_tokens=10):
    """
    Classify a single incident using the provided LLM model with few-shot prompting.
    """
    # Handle missing values
    if pd.isna(title):
        title = ""
    if pd.isna(description):
        description = ""
    
    title_str = str(title).lower()
    desc_str = str(description).lower()
    combined_text = title_str + " " + desc_str
    
    # Rule-based pre-classification for obvious cases (improves recall)
    ps_keyword_count = count_ps_keywords(combined_text, language)
    
    # If multiple PS keywords present, classify as PS directly (high confidence)
    if ps_keyword_count >= 3:
        return "Process Safety"
    
    # Build prompt
    prompt = build_few_shot_prompt(str(title), str(description), language)
    
    # Tokenize
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        max_length=max_length, 
        truncation=True
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1
        )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Parse response - more inclusive for Process Safety
    response_lower = response.lower()
    
    # Check for Non-Process Safety first (must be explicit)
    if "non-process" in response_lower or "non process" in response_lower:
        # Double-check: if keywords present, override to PS
        if ps_keyword_count >= 2:
            return "Process Safety"
        return "Non-Process Safety"
    elif "process" in response_lower or "safety" in response_lower:
        return "Process Safety"
    else:
        # Default: if any PS keywords, classify as PS
        if ps_keyword_count >= 1:
            return "Process Safety"
        return "Non-Process Safety"

# Print summary
print("[OK] Comprehensive Process Safety Keywords loaded (CCPS-Based)")
print(f"[INFO] Keywords by language:")
for lang, keywords in PS_KEYWORDS.items():
    print(f"  {lang}: {len(keywords)} keywords")
print(f"[INFO] Total unique keywords (all languages): {len(ALL_PS_KEYWORDS)}")

[OK] Comprehensive Process Safety Keywords loaded (CCPS-Based)
[INFO] Keywords by language:
  English: 316 keywords
  German: 293 keywords
  Dutch: 301 keywords
  Swedish: 298 keywords
[INFO] Total unique keywords (all languages): 1064


In [7]:
# =============================================================================
# BATCH CLASSIFICATION PIPELINE WITH MODEL PARAMETER
# =============================================================================

def run_llm_classification_pipeline_for_model(df, model_name, tokenizer, model, device, batch_save_interval=100):
    """
    Run LLM classification on entire dataframe with progress tracking and checkpoint.
    """
    results = []
    # Use model-specific checkpoint file
    safe_model_name = model_name.replace("/", "_").replace("-", "_")
    checkpoint_file = os.path.join(RESULTS_PATH, f'classification_checkpoint_{safe_model_name}.csv')
    
    # Check for existing checkpoint
    start_idx = 0
    if os.path.exists(checkpoint_file):
        checkpoint_df = pd.read_csv(checkpoint_file)
        start_idx = len(checkpoint_df)
        results = checkpoint_df['LLM_Classification'].tolist()
        print(f"[INFO] Resuming from checkpoint at index {start_idx}")
    
    total_records = len(df)
    print(f"[INFO] Model: {model_name}")
    print(f"[INFO] Total records: {total_records}, Starting from: {start_idx}")
    print(f"[INFO] Checkpoint interval: {batch_save_interval}")
    
    start_time = time.time()
    ps_count = sum(1 for r in results if r == "Process Safety")
    
    # Single progress bar
    pbar = tqdm(range(start_idx, total_records), desc=f"Classifying ({model_name.split('/')[-1]})", unit="rec")
    
    for idx in pbar:
        row = df.iloc[idx]
        title = row['TITLE']
        description = row['CASE_DESCRIPTION']
        language = row.get('LANGUAGE', 'English')
        
        try:
            classification = classify_incident_with_model(title, description, tokenizer, model, device, language)
        except Exception as e:
            # On error, use keyword-based fallback
            combined = str(title).lower() + " " + str(description).lower()
            if any(kw in combined for kw in PS_KEYWORDS[:20]):
                classification = "Process Safety"
            else:
                classification = "Non-Process Safety"
        
        if classification == "Process Safety":
            ps_count += 1
        
        results.append(classification)
        
        # Update progress bar with PS count
        pbar.set_postfix({'PS': ps_count, 'NPS': len(results) - ps_count})
        
        # Save checkpoint silently
        if (idx + 1) % batch_save_interval == 0:
            temp_df = df.iloc[:idx+1].copy()
            temp_df['LLM_Classification'] = results
            temp_df.to_csv(checkpoint_file, index=False)
            # Clear CUDA cache periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    pbar.close()
    
    # Final save
    df_result = df.copy()
    df_result['LLM_Classification'] = results
    df_result.to_csv(checkpoint_file, index=False)
    
    elapsed_total = time.time() - start_time
    print(f"[OK] Classification complete for {model_name}!")
    print(f"  Time: {elapsed_total/60:.2f} min")
    if (total_records - start_idx) > 0:
        print(f"  Rate: {(total_records - start_idx)/elapsed_total:.2f} rec/s")
    print(f"  Process Safety: {ps_count}")
    print(f"  Non-Process Safety: {len(results) - ps_count}")
    
    return results

print("[OK] Batch classification pipeline defined with model parameter")

[OK] Batch classification pipeline defined with model parameter


In [8]:
# =============================================================================
# RUN CLASSIFICATION FOR BOTH MODELS
# =============================================================================

for model_name in MODELS_TO_COMPARE:
    print("\n" + "=" * 70)
    print(f"  RUNNING CLASSIFICATION WITH: {model_name}")
    print("=" * 70)
    
    # Load model
    print(f"\n[INFO] Loading model: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model = model.to(device)
    model.eval()
    print(f"[OK] Model loaded: {model_name}")
    print(f"[INFO] Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Run classification
    print(f"\n[INFO] Starting classification pipeline for {model_name}...")
    classifications = run_llm_classification_pipeline_for_model(
        master_df_7, model_name, tokenizer, model, device, batch_save_interval=100
    )
    
    # Store results
    all_model_results[model_name] = classifications
    
    # Save results to CSV
    safe_model_name = model_name.replace("/", "_").replace("-", "_")
    result_df = master_df_7.copy()
    result_df['LLM_Classification'] = classifications
    result_path = os.path.join(RESULTS_PATH, f'master_df_7_{safe_model_name}.csv')
    result_df.to_csv(result_path, index=False)
    print(f"[OK] Results saved to: {result_path}")
    
    # Clear memory
    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f"[INFO] Model unloaded and memory cleared")

print("\n" + "=" * 70)
print("  ALL MODEL CLASSIFICATIONS COMPLETE")
print("=" * 70)


  RUNNING CLASSIFICATION WITH: google/flan-t5-large

[INFO] Loading model: google/flan-t5-large...
[OK] Model loaded: google/flan-t5-large
[INFO] Model parameters: 783,150,080

[INFO] Starting classification pipeline for google/flan-t5-large...
[INFO] Resuming from checkpoint at index 46000
[INFO] Model: google/flan-t5-large
[INFO] Total records: 60968, Starting from: 46000
[INFO] Checkpoint interval: 100


Classifying (flan-t5-large): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14968/14968 [2:20:21<00:00,  1.78rec/s, PS=18865, NPS=42103]


[OK] Classification complete for google/flan-t5-large!
  Time: 140.37 min
  Rate: 1.78 rec/s
  Process Safety: 18865
  Non-Process Safety: 42103
[OK] Results saved to: /home/azureuser/cloudfiles/code/Users/M02555/Results/_iteration_7/master_df_7_google_flan_t5_large.csv
[INFO] Model unloaded and memory cleared

  RUNNING CLASSIFICATION WITH: google/flan-t5-xl

[INFO] Loading model: google/flan-t5-xl...


RuntimeError: Data processing error: CAS service error : IO Error: No space left on device (os error 28)

In [ ]:
# =============================================================================
# EVALUATE BOTH MODELS AND COMPARE
# =============================================================================

def evaluate_model_classification(df, classifications, model_name):
    """
    Evaluate classification results for a specific model.
    """
    # Convert ground truth to binary: 1 for Process Safety, 0 for Non-Process Safety
    y_true = df['CASE_TYPE'].apply(lambda x: 1 if x == 'Process Safety' else 0)
    y_pred = pd.Series(classifications).apply(lambda x: 1 if x == 'Process Safety' else 0)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_ps = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_ps = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_ps = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    
    precision_nps = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    recall_nps = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1_nps = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    
    metrics = {
        'model_name': model_name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'precision_ps': precision_ps,
        'recall_ps': recall_ps,
        'f1_ps': f1_ps,
        'precision_nps': precision_nps,
        'recall_nps': recall_nps,
        'f1_nps': f1_nps,
        'confusion_matrix': cm,
        'false_positives': cm[1,0],
        'false_negatives': cm[0,1],
        'true_positives': cm[0,0],
        'true_negatives': cm[1,1]
    }
    
    return metrics

# Evaluate all models
print("\n" + "=" * 70)
print("                    EVALUATING ALL MODELS")
print("=" * 70)

for model_name, classifications in all_model_results.items():
    metrics = evaluate_model_classification(master_df_7, classifications, model_name)
    all_model_metrics[model_name] = metrics
    
    print(f"\n{'='*70}")
    print(f"[{model_name}]")
    print(f"{'='*70}")
    print(f"  Accuracy:     {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    print(f"  Macro F1:     {metrics['macro_f1']:.4f} ({metrics['macro_f1']*100:.2f}%)")
    print(f"  PS Precision: {metrics['precision_ps']:.4f}")
    print(f"  PS Recall:    {metrics['recall_ps']:.4f}")
    print(f"  PS F1:        {metrics['f1_ps']:.4f}")
    print(f"  NPS Precision:{metrics['precision_nps']:.4f}")
    print(f"  NPS Recall:   {metrics['recall_nps']:.4f}")
    print(f"  NPS F1:       {metrics['f1_nps']:.4f}")
    
    # Print confusion matrix for each model
    print(f"\n  CONFUSION MATRIX for {model_name.split('/')[-1]}:")
    print(f"  {'-'*50}")
    print(f"                           Predicted PS | Predicted NPS")
    print(f"      Actual PS               {metrics['true_positives']:5d}    |    {metrics['false_negatives']:5d}")
    print(f"      Actual NPS              {metrics['false_positives']:5d}    |    {metrics['true_negatives']:5d}")
    print(f"  {'-'*50}")
    print(f"  Raw Confusion Matrix (rows=Actual, cols=Predicted):")
    print(f"  {metrics['confusion_matrix']}")
    print()


                    EVALUATING ALL MODELS

[google/flan-t5-large]
  Accuracy:     0.7227 (72.27%)
  Macro F1:     0.6130 (61.30%)
  PS Precision: 0.3076
  PS Recall:    0.6013
  PS F1:        0.4070
  NPS Precision:0.9086
  NPS Recall:   0.7455
  NPS F1:       0.8190
  Confusion Matrix:
                     Predicted PS | Predicted NPS
    Actual PS            5803    |     3847
    Actual NPS          13062    |    38256


In [ ]:
# =============================================================================
# VISUALIZE CONFUSION MATRIX FOR EACH MODEL (INDIVIDUAL PLOTS)
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

print("\n" + "=" * 70)
print("        CONFUSION MATRIX VISUALIZATIONS (INDIVIDUAL)")
print("=" * 70)

for model_name, metrics in all_model_metrics.items():
    cm = metrics['confusion_matrix']
    short_name = model_name.split('/')[-1]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['PS (1)', 'NPS (0)'],
                yticklabels=['PS (1)', 'NPS (0)'],
                annot_kws={'size': 14})
    axes[0].set_title(f'{short_name}\nConfusion Matrix (Counts)\nMacro F1: {metrics["macro_f1"]*100:.1f}%', 
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('Actual', fontsize=11)
    
    # Normalized (percentages)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
                xticklabels=['PS (1)', 'NPS (0)'],
                yticklabels=['PS (1)', 'NPS (0)'],
                annot_kws={'size': 14})
    axes[1].set_title(f'{short_name}\nConfusion Matrix (% per Actual Class)', 
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted', fontsize=11)
    axes[1].set_ylabel('Actual', fontsize=11)
    
    plt.suptitle(f'Model: {model_name}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save individual figure
    safe_name = model_name.replace("/", "_").replace("-", "_")
    fig_path = os.path.join(RESULTS_PATH, f'confusion_matrix_{safe_name}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"\n[OK] Confusion matrix saved: {fig_path}")
    
    plt.show()
    print()

In [10]:
# =============================================================================
# MODEL COMPARISON SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("                    MODEL COMPARISON SUMMARY")
print("=" * 70)

# Create comparison DataFrame
comparison_data = []
for model_name, metrics in all_model_metrics.items():
    comparison_data.append({
        'Model': model_name.split('/')[-1],
        'Accuracy': f"{metrics['accuracy']*100:.2f}%",
        'Macro F1': f"{metrics['macro_f1']*100:.2f}%",
        'PS Precision': f"{metrics['precision_ps']*100:.2f}%",
        'PS Recall': f"{metrics['recall_ps']*100:.2f}%",
        'PS F1': f"{metrics['f1_ps']*100:.2f}%",
        'NPS Precision': f"{metrics['precision_nps']*100:.2f}%",
        'NPS Recall': f"{metrics['recall_nps']*100:.2f}%",
        'NPS F1': f"{metrics['f1_nps']*100:.2f}%",
        'False Pos': metrics['false_positives'],
        'False Neg': metrics['false_negatives']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n[COMPARISON TABLE]")
print(comparison_df.to_string(index=False))

# Determine the best model based on Macro F1
best_model = max(all_model_metrics.items(), key=lambda x: x[1]['macro_f1'])
print(f"\n[BEST MODEL BY MACRO F1]")
print(f"  Winner: {best_model[0]}")
print(f"  Macro F1: {best_model[1]['macro_f1']*100:.2f}%")

# Determine best by PS Recall (important for catching all Process Safety incidents)
best_recall = max(all_model_metrics.items(), key=lambda x: x[1]['recall_ps'])
print(f"\n[BEST MODEL BY PS RECALL]")
print(f"  Winner: {best_recall[0]}")
print(f"  PS Recall: {best_recall[1]['recall_ps']*100:.2f}%")

# Save comparison to CSV
comparison_df.to_csv(os.path.join(RESULTS_PATH, 'model_comparison.csv'), index=False)
print(f"\n[OK] Comparison saved to: {RESULTS_PATH}model_comparison.csv")


                    MODEL COMPARISON SUMMARY

[COMPARISON TABLE]
        Model Accuracy Macro F1 PS Precision PS Recall  PS F1 NPS Precision NPS Recall NPS F1  False Pos  False Neg
flan-t5-large   72.27%   61.30%       30.76%    60.13% 40.70%        90.86%     74.55% 81.90%      13062       3847

[BEST MODEL BY MACRO F1]
  Winner: google/flan-t5-large
  Macro F1: 61.30%

[BEST MODEL BY PS RECALL]
  Winner: google/flan-t5-large
  PS Recall: 60.13%

[OK] Comparison saved to: /home/azureuser/cloudfiles/code/Users/M02555/Results/_iteration_7/model_comparison.csv


In [ ]:
# =============================================================================
# CONFUSION MATRIX VISUALIZATION FOR BOTH MODELS
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrices_comparison(all_metrics, save_path):
    """
    Create confusion matrix plots for all models side by side.
    """
    n_models = len(all_metrics)
    fig, axes = plt.subplots(2, n_models, figsize=(7*n_models, 10))
    
    if n_models == 1:
        axes = axes.reshape(2, 1)
    
    for idx, (model_name, metrics) in enumerate(all_metrics.items()):
        cm = metrics['confusion_matrix']
        short_name = model_name.split('/')[-1]
        
        # Raw counts
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, idx],
                    xticklabels=['PS (1)', 'NPS (0)'],
                    yticklabels=['PS (1)', 'NPS (0)'])
        axes[0, idx].set_title(f'{short_name}\nCounts (Macro F1: {metrics["macro_f1"]*100:.1f}%)', 
                               fontsize=11, fontweight='bold')
        axes[0, idx].set_xlabel('Predicted')
        axes[0, idx].set_ylabel('Actual')
        
        # Normalized (percentages)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
        sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='Blues', ax=axes[1, idx],
                    xticklabels=['PS (1)', 'NPS (0)'],
                    yticklabels=['PS (1)', 'NPS (0)'])
        axes[1, idx].set_title(f'{short_name}\n% per Actual Class', fontsize=11, fontweight='bold')
        axes[1, idx].set_xlabel('Predicted')
        axes[1, idx].set_ylabel('Actual')
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(save_path, 'confusion_matrix_comparison.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"[OK] Confusion matrix comparison saved to: {fig_path}")
    
    plt.show()

# Plot confusion matrices for all models
plot_confusion_matrices_comparison(all_model_metrics, RESULTS_PATH)

In [ ]:
# =============================================================================
# SAVE FINAL METRICS SUMMARY FOR ALL MODELS
# =============================================================================

def save_all_models_summary(all_metrics, df, save_path):
    """
    Save comprehensive metrics summary for all models to file and print to notebook.
    """
    summary_path = os.path.join(save_path, 'llm_classification_summary_all_models.txt')
    
    # Build summary content
    summary_lines = []
    summary_lines.append("=" * 80)
    summary_lines.append("     LLM CLASSIFICATION SUMMARY - ITERATION 7 (MODEL COMPARISON)")
    summary_lines.append("=" * 80)
    summary_lines.append("")
    summary_lines.append("[DATASET INFORMATION]")
    summary_lines.append(f"  Total Records: {len(df)}")
    summary_lines.append(f"  Process Safety (Actual): {len(df[df['CASE_TYPE']=='Process Safety'])}")
    summary_lines.append(f"  Non-Process Safety (Actual): {len(df[df['CASE_TYPE']!='Process Safety'])}")
    summary_lines.append(f"  Languages: English, German, Dutch, Swedish")
    summary_lines.append("")
    
    for model_name, metrics in all_metrics.items():
        summary_lines.append("=" * 80)
        summary_lines.append(f"  MODEL: {model_name}")
        summary_lines.append("=" * 80)
        summary_lines.append("")
        summary_lines.append("[CLASSIFICATION METRICS]")
        summary_lines.append(f"  Accuracy:     {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
        summary_lines.append(f"  Macro F1:     {metrics['macro_f1']:.4f} ({metrics['macro_f1']*100:.2f}%)")
        summary_lines.append("")
        summary_lines.append("[PROCESS SAFETY METRICS]")
        summary_lines.append(f"  Precision: {metrics['precision_ps']:.4f}")
        summary_lines.append(f"  Recall:    {metrics['recall_ps']:.4f}")
        summary_lines.append(f"  F1-Score:  {metrics['f1_ps']:.4f}")
        summary_lines.append("")
        summary_lines.append("[NON-PROCESS SAFETY METRICS]")
        summary_lines.append(f"  Precision: {metrics['precision_nps']:.4f}")
        summary_lines.append(f"  Recall:    {metrics['recall_nps']:.4f}")
        summary_lines.append(f"  F1-Score:  {metrics['f1_nps']:.4f}")
        summary_lines.append("")
        summary_lines.append("[CONFUSION MATRIX]")
        summary_lines.append("                           Predicted")
        summary_lines.append("                    PS (1)     NPS (0)")
        summary_lines.append(f"  Actual PS (1)     {metrics['confusion_matrix'][0,0]:5d}      {metrics['confusion_matrix'][0,1]:5d}")
        summary_lines.append(f"  Actual NPS (0)    {metrics['confusion_matrix'][1,0]:5d}      {metrics['confusion_matrix'][1,1]:5d}")
        summary_lines.append("")
        summary_lines.append("[ERROR ANALYSIS]")
        summary_lines.append(f"  False Positives (predicted PS, actual NPS): {metrics['false_positives']}")
        summary_lines.append(f"  False Negatives (predicted NPS, actual PS): {metrics['false_negatives']}")
        summary_lines.append("")
    
    # Add comparison section
    summary_lines.append("=" * 80)
    summary_lines.append("                         MODEL COMPARISON")
    summary_lines.append("=" * 80)
    best_macro_f1 = max(all_metrics.items(), key=lambda x: x[1]['macro_f1'])
    best_recall = max(all_metrics.items(), key=lambda x: x[1]['recall_ps'])
    summary_lines.append(f"  Best by Macro F1:  {best_macro_f1[0]} ({best_macro_f1[1]['macro_f1']*100:.2f}%)")
    summary_lines.append(f"  Best by PS Recall: {best_recall[0]} ({best_recall[1]['recall_ps']*100:.2f}%)")
    summary_lines.append("")
    summary_lines.append("=" * 80)
    
    # Print to notebook
    print("\n".join(summary_lines))
    
    # Save to file
    with open(summary_path, 'w') as f:
        f.write("\n".join(summary_lines))
    
    print(f"\n[OK] Summary saved to: {summary_path}")

# Save and print summary for all models
save_all_models_summary(all_model_metrics, master_df_7, RESULTS_PATH)

print("\n" + "=" * 70)
print("           ITERATION 7 LLM CLASSIFICATION PIPELINE COMPLETE")
print("=" * 70)
print(f"\n[OUTPUT FILES]")
print(f"  1. {RESULTS_PATH}model_comparison.csv")
print(f"  2. {RESULTS_PATH}confusion_matrix_comparison.png")
print(f"  3. {RESULTS_PATH}llm_classification_summary_all_models.txt")
for model_name in MODELS_TO_COMPARE:
    safe_name = model_name.replace("/", "_").replace("-", "_")
    print(f"  4. {RESULTS_PATH}master_df_7_{safe_name}.csv")

In [ ]:
# =============================================================================
# BAR CHART COMPARISON OF KEY METRICS
# =============================================================================

def plot_metrics_comparison_bar(all_metrics, save_path):
    """
    Create bar charts comparing key metrics across models.
    """
    models = [m.split('/')[-1] for m in all_metrics.keys()]
    
    metrics_to_plot = {
        'Macro F1': [m['macro_f1']*100 for m in all_metrics.values()],
        'PS Recall': [m['recall_ps']*100 for m in all_metrics.values()],
        'PS Precision': [m['precision_ps']*100 for m in all_metrics.values()],
        'Accuracy': [m['accuracy']*100 for m in all_metrics.values()],
    }
    
    x = np.arange(len(models))
    width = 0.2
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
    
    for i, (metric_name, values) in enumerate(metrics_to_plot.items()):
        bars = ax.bar(x + i*width, values, width, label=metric_name, color=colors[i])
        # Add value labels on bars
        for bar, val in zip(bars, values):
            ax.annotate(f'{val:.1f}%',
                       xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=9)
    
    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title('Model Comparison: Key Classification Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width*1.5)
    ax.set_xticklabels(models, fontsize=11)
    ax.legend(loc='lower right', fontsize=10)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    fig_path = os.path.join(save_path, 'model_comparison_bar_chart.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"[OK] Bar chart comparison saved to: {fig_path}")
    
    plt.show()

# Plot bar chart comparison
plot_metrics_comparison_bar(all_model_metrics, RESULTS_PATH)

# Final recommendation
print("\n" + "=" * 70)
print("                         RECOMMENDATION")
print("=" * 70)
best_model = max(all_model_metrics.items(), key=lambda x: x[1]['macro_f1'])
print(f"\nBased on Macro F1 Score, the recommended model is:")
print(f"  >>> {best_model[0]} <<<")
print(f"  Macro F1: {best_model[1]['macro_f1']*100:.2f}%")
print(f"  PS Recall: {best_model[1]['recall_ps']*100:.2f}%")
print(f"  PS Precision: {best_model[1]['precision_ps']*100:.2f}%")
print("\n" + "=" * 70)